# 13 - Orçamento de Incerteza: Quanto Confiar em um Registro

Os notebooks 01 a 12 trataram o banco de dados como se todos os registros
merecessem a mesma confiança. Este notebook pergunta o quanto uma espécie pode
se afastar da termoquímica de referência. Desde a versão 0.5.0, o `pyglenn`
traz um envelope de erro por espécie, exposto por
`ThermochemicalCalculator.uncertainty`. O envelope resume, em um conjunto de
números independente da temperatura, a diferença entre os polinômios da NASA
empacotados e as tabelas NIST-JANAF: um viés e uma banda de dois sigma para Cp e
H, além de uma banda absoluta de dois sigma para S.

O roteiro é curto. Começamos pelo que o método devolve, medimos quanto do
catálogo está de fato validado, examinamos as espécies que o envelope sinaliza e
terminamos propagando a banda para uma entalpia de reação.

In [1]:
import pandas as pd

from pyglenn import ThermochemicalCalculator

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


## 1. O que o método devolve

`uncertainty(species_id, temperature)` devolve um dicionário. A temperatura é
verificada contra a faixa da espécie mas não altera os números, porque o
envelope é um único resumo para todo o intervalo.

| chave | significado |
|-----|---------|
| `cp_rel_bias` | desvio relativo médio de Cp, em porcentagem |
| `cp_rel_2sigma` | banda relativa de dois sigma de Cp, em porcentagem |
| `h_rel_bias` | desvio relativo médio da entalpia padronizada, em porcentagem |
| `h_rel_2sigma` | banda relativa de dois sigma de H, em porcentagem |
| `s_abs_2sigma` | banda de dois sigma de S, absoluta, em J/(mol K) |
| `n_ref_points` | número de pontos de referência por trás do envelope |
| `source` | `nist-janaf` para espécies validadas, `extrapolated` caso contrário |

In [2]:
with ThermochemicalCalculator() as calc:
    co2 = calc.get_available_species("CO2", exact_match=True)[0]
    envelope = calc.uncertainty(co2.id, 1000.0)

for key, value in envelope.items():
    print(f"{key:15s}: {value}")

species_id     : 316
species_name   : CO2
temperature    : 1000.0
cp_rel_bias    : 0.1874
cp_rel_2sigma  : 1.4138
h_rel_bias     : 0.0221
h_rel_2sigma   : 0.1668
s_abs_2sigma   : 0.0912
n_ref_points   : 16
source         : nist-janaf


## 2. Quanto do catálogo está validado

O rótulo em `source` importa mais do que os números. Uma espécie validada foi
comparada ponto a ponto com dados tabelados de referência. O restante carrega uma
banda padrão que veio do estudo de validação, não de um ajuste a essa espécie
específica.

In [3]:
with ThermochemicalCalculator() as calc:
    catalogue = calc.get_available_species("")
    sources = {}
    for s in catalogue:
        low, high = calc.db.get_temperature_bounds(s.id)
        env = calc.uncertainty(s.id, 0.5 * (low + high))
        label = "missing" if env is None else env["source"]
        sources[label] = sources.get(label, 0) + 1

for label, count in sources.items():
    print(f"{label:12s}: {count:5d}")

extrapolated:  2017
nist-janaf  :    18


O desequilíbrio é grande. Apenas dezoito espécies foram validadas à mão e as outras
dois mil e dezessete herdam o envelope padrão. Não é um defeito da
biblioteca, é uma afirmação sobre o custo de validar, e deveria moderar qualquer
alegação de exatidão para uma espécie arbitrária.

## 3. Espécies sinalizadas pelo envelope

O envelope se justifica como ferramenta de triagem. Entre as espécies validadas
usadas neste notebook, o dióxido de carbono fica em 1.4 por cento em Cp, o
oxigênio em 0.8 e o nitrogênio em 0.2; a água chega a 3.1, e as entradas
extrapoladas carregam a banda padrão de cerca de 9 por cento.

In [4]:
names = ["CO2", "H2O", "O2", "N2", "CH4", "C2H5OH"]

rows = []
with ThermochemicalCalculator() as calc:
    for name in names:
        s = calc.get_available_species(name, exact_match=True)[0]
        env = calc.uncertainty(s.id, 1000.0)
        rows.append({
            "species": s.name,
            "source": env["source"],
            "Cp bias [%]": env["cp_rel_bias"],
            "Cp 2sigma [%]": env["cp_rel_2sigma"],
            "H 2sigma [%]": env["h_rel_2sigma"],
            "S 2sigma [J/mol/K]": env["s_abs_2sigma"],
        })

pd.DataFrame(rows).set_index("species").round(2)

,source,Cp bias [%],Cp 2sigma [%],H 2sigma [%],S 2sigma [J/mol/K]
species,,,,,
CO2,nist-janaf,0.190,1.410,0.170,0.090
H2O,nist-janaf,1.100,3.100,1.630,1.170
O2,nist-janaf,0.100,0.770,0.280,0.150
N2,nist-janaf,0.030,0.180,0.020,0.010
CH4,nist-janaf,8.820,27.610,13.550,17.290
C2H5OH,extrapolated,0.000,9.070,4.300,5.100


O metano é a exceção, com 27.6 por cento, uma ordem de grandeza acima das
demais. Um
sinal desse tamanho não prova que o registro esteja errado, mas estabelece onde
olhar primeiro. A seção seguinte segue essa pista.

## 4. Acompanhamento de uma espécie sinalizada

A primeira pergunta diante de um registro sinalizado é se as âncoras
fundamentais se sustentam. Três valores bastam: a capacidade calorífica, a
entropia e a entalpia de formação a 298.15 K. Os valores de referência do metano
são amplamente tabelados e a comparação exige um único laço.

In [5]:
anchors = {                      # NIST reference values for CH4(g)
    "Cp(298.15 K) [J/mol/K]": 35.69,
    "S(298.15 K) [J/mol/K]": 186.25,
    "dHf(298.15 K) [kJ/mol]": -74.87,
}

with ThermochemicalCalculator() as calc:
    ch4 = calc.get_available_species("CH4", exact_match=True)[0]
    props = calc.calculate_properties(ch4.id, 298.15)
    computed = {
        "Cp(298.15 K) [J/mol/K]": props.cp,
        "S(298.15 K) [J/mol/K]": props.s,
        "dHf(298.15 K) [kJ/mol]": props.h_relative / 1000.0,
    }

for label, reference in anchors.items():
    print(f"{label:24s} pyglenn {computed[label]:9.3f}   NIST {reference:9.3f}")

Cp(298.15 K) [J/mol/K]   pyglenn    35.691   NIST    35.690
S(298.15 K) [J/mol/K]    pyglenn   186.371   NIST   186.250
dHf(298.15 K) [kJ/mol]   pyglenn   -74.600   NIST   -74.870


As três âncoras ficam a décimos de porcentagem da referência, então o registro
é sólido perto da temperatura ambiente. A banda sinalizada se concentra, assim,
em alta temperatura e nos desvios de entropia e entalpia, não na forma do
polinômio que mais usamos. A lição é de procedimento: um envelope é um ponteiro,
e o ponteiro precisa ser seguido por uma verificação independente antes de
declarar que algo está quebrado.

## 5. Propagação da banda para uma entalpia de reação

Para uma combinação linear, como uma entalpia de reação, as bandas se combinam em
quadratura com os coeficientes estequiométricos como pesos. O exemplo abaixo usa
a combustão completa do metano a 298.15 K e trata a banda relativa de H como se
valesse para a entalpia padronizada de cada espécie.

In [6]:
reaction = {"CH4": -1.0, "O2": -2.0, "CO2": +1.0, "H2O": +2.0}

terms = []
with ThermochemicalCalculator() as calc:
    for name, nu in reaction.items():
        s = calc.get_available_species(name, exact_match=True)[0]
        h = calc.calculate_properties(s.id, 298.15).h_relative / 1000.0
        env = calc.uncertainty(s.id, 298.15)
        terms.append({
            "species": name,
            "nu": nu,
            "H [kJ/mol]": h,
            "2sigma [kJ/mol]": abs(h) * env["h_rel_2sigma"] / 100.0,
        })

table = pd.DataFrame(terms).set_index("species")
dH = float((table["nu"] * table["H [kJ/mol]"]).sum())
band = float(((table["nu"] * table["2sigma [kJ/mol]"]) ** 2).sum() ** 0.5)

print(table.round(2))
print(f"\nDelta H_rxn(298.15 K) = {dH:8.1f} kJ/mol")
print(f"combined 2sigma       = {band:8.1f} kJ/mol  ({100 * band / abs(dH):.1f} % of the enthalpy)")

            nu  H [kJ/mol]  2sigma [kJ/mol]
species                                    
CH4     -1.000     -74.600           10.110
O2      -2.000      -0.000            0.000
CO2      1.000    -393.510            0.660
H2O      2.000    -241.830            3.940

Delta H_rxn(298.15 K) =   -802.6 kJ/mol
combined 2sigma       =     12.8 kJ/mol  (1.6 % of the enthalpy)


A banda combinada de cerca de 13 kJ/mol corresponde a 1.6 por cento da entalpia
de reação, e quase toda ela vem do combustível. Essa assimetria é típica: a
espécie com maior entalpia de formação domina o orçamento, de modo que um
combustível validado vale mais do que um produto validado.

## 6. O que fazer com isso

Confira o rótulo de origem antes de confiar em uma espécie que sustenta uma
conclusão. Se o registro é validado, cite a banda junto ao valor calculado. Se é
extrapolado, diga isso. Se uma banda parecer anômala, compare a espécie com uma
tabela independente antes de usá-la em um modelo quantitativo. O envelope reduz
a busca, não substitui a referência.